In [1]:
from ccdc import io
from ccdc.io import EntryReader

from itertools import islice

from tqdm import tqdm

import pandas as pd
import time

In [2]:
csd = EntryReader('CSD')

### 1. Extract all records to single CSV table

In [ ]:
all_df = []

start = time.time()

total_entry = 500
for entry in tqdm(islice(csd, total_entry), total=total_entry, desc="Processing"):

    # check organic
    is_organic = entry.is_organic

    # check num components
    num_component = len(entry.molecule.components)

    # check metal
    has_metal = any(atom.is_metal for atom in entry.molecule.atoms)

    # add result
    all_df.append({
        "ID":entry.identifier,
        "SMILES": entry.molecule.smiles,
        "IS_ORGANIC": is_organic,
        "HAS_METAL":has_metal,
        "NUM_COMPONENT": num_component,
    })

delta_time = round(time.time() - start, 1)

all_df = pd.DataFrame(all_df)
all_df.to_csv("csd_all.csv", index=False)

print(f"Sequential : Processed {len(all_df)//1000}k entries in {delta_time}s")

In [11]:
all_df

,ID,SMILES,IS_ORGANIC,HAS_METAL,NUM_COMPONENT
0,AABHTZ,CC(=O)NN1C=NN=C1N(N=Cc1c(Cl)cccc1Cl)C(C)=O,True,False,1
1,AACANI10,[OH2][Ni]123OC(=O)CN41CCCN2(CCC4)CC(=O)O3.O.O,False,True,3
2,AACANI11,[OH2][Ni]123OC(=O)CN41CCCN2(CCC4)CC(=O)O3.O.O,False,True,3
3,AACFAZ,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
4,AACFAZ10,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
...,...,...,...,...,...
495,ABAZPT,CC(=O)N1=NC(=[NH][Pt]21[NH]=C(N=N2C(C)=O)C(C)(...,False,True,1
496,ABAZUA,Cc1cc(C)c(c(C)c1)B1(c2ccccc2c2ccc(cn12)c1ccc(c...,True,False,1
497,ABAZUB,O.COc1cc(Nc2c(cnc3cc(OCCCN4CCN(C)CC4)c(OC)cc23...,True,False,2
498,ABAZUB01,COc1cc(Nc2c(cnc3cc(OCCCN4CCN(C)CC4)c(OC)cc23)C...,True,False,2


In [ ]:
all_df

### 2. Select only organic one component molecules

In [3]:
from CSDTools.processing import filter_entries

all_df = pd.read_csv("csd_all.csv")
df_filtered = filter_entries(all_df)
print(f"Selected {len(df_filtered)} entries")

Selected 445735 entries


In [4]:
df_filtered

,identifier,chemical_name,is_racemic_name,is_organic,has_metal,num_components,is_single_species,has_disorder,disorder_details,color,...,cell_angle_alpha,cell_angle_beta,cell_angle_gamma,cell_volume,reduced_cell_a,reduced_cell_b,reduced_cell_c,reduced_cell_alpha,reduced_cell_beta,reduced_cell_gamma
0,AABHTZ,"4-Acetoamido-3-(1-acetyl-2-(2,6-dichlorobenzyl...",False,True,False,1,True,False,NaN,NaN,...,108.750,71.070,96.160,769.978363,7.3590,10.2720,11.365707,84.217121,71.162715,71.250
3,AACFAZ,anti-anti-bis(2-o-Chlorophenyl-4-methoxy-5-oxo...,False,True,False,1,True,False,NaN,red,...,90.000,90.000,90.000,2475.361224,6.1620,19.9540,20.132000,90.000000,90.000000,90.000
4,AACFAZ10,"N,N'-bis(3-Acetyl-4-(2-chlorophenyl)-4-hydroxy...",False,True,False,1,True,False,NaN,red,...,90.000,90.000,90.000,2475.361224,6.1620,19.9540,20.132000,90.000000,90.000000,90.000
6,AACMHX10,"α-Acetoxy-α,2-anti-diphenylmethylene-cyclohexane",False,True,False,1,True,False,NaN,NaN,...,90.000,90.000,90.000,3455.164356,8.5350,16.7580,24.157000,90.000000,90.000000,90.000
14,AADRIB,"1,2,3,4-Tetra-O-acetyl-α-D-ribopyranose",False,True,False,2,True,False,NaN,NaN,...,90.000,109.180,90.000,1591.259021,8.5900,11.4530,16.443713,90.000000,100.383702,90.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1436108,ZUYNAM01,5-(pentafluoro-λ6-sulfanyl)pentyl 4-t-butylben...,False,True,False,1,True,True,C00L and C00M disordered over two sites with o...,light colorless,...,90.000,90.000,90.000,935.014010,7.0260,7.6080,17.492000,90.000000,90.000000,90.000
1436112,ZZZEIG05,triphenylarsane,False,True,False,4,True,False,NaN,whiteish colorless,...,84.551,80.072,86.361,2999.830947,11.2004,15.2918,17.882400,84.551000,80.072000,86.361
1436113,ZZZEIG06,triphenylarsane,False,True,False,1,True,False,NaN,whiteish colorless,...,84.551,80.072,86.361,2999.830947,11.2004,15.2918,17.882400,84.551000,80.072000,86.361
1436114,ZZZOLM02,docosanoic acid,False,True,False,1,True,False,NaN,colorless,...,90.000,90.052,90.000,2089.560807,5.5210,7.1540,52.904000,90.000000,90.052000,90.000


### 3. Count the number of crystal forms

In [5]:
print(f"Unique molecules: {df_filtered['inchi_single'].nunique():,}")


Unique molecules: 382878


In [6]:
df_counted

,identifier,chemical_name,is_racemic_name,is_organic,has_metal,num_components,is_single_species,has_disorder,disorder_details,color,...,cell_angle_beta,cell_angle_gamma,cell_volume,reduced_cell_a,reduced_cell_b,reduced_cell_c,reduced_cell_alpha,reduced_cell_beta,reduced_cell_gamma,num_forms
0,AABHTZ,"4-Acetoamido-3-(1-acetyl-2-(2,6-dichlorobenzyl...",False,True,False,1,True,False,NaN,NaN,...,71.070,96.160,769.978363,7.3590,10.2720,11.365707,84.217121,71.162715,71.250,1
4,AACFAZ10,"N,N'-bis(3-Acetyl-4-(2-chlorophenyl)-4-hydroxy...",False,True,False,1,True,False,NaN,red,...,90.000,90.000,2475.361224,6.1620,19.9540,20.132000,90.000000,90.000000,90.000,1
6,AACMHX10,"α-Acetoxy-α,2-anti-diphenylmethylene-cyclohexane",False,True,False,1,True,False,NaN,NaN,...,90.000,90.000,3455.164356,8.5350,16.7580,24.157000,90.000000,90.000000,90.000,1
14,AADRIB,"1,2,3,4-Tetra-O-acetyl-α-D-ribopyranose",False,True,False,2,True,False,NaN,NaN,...,109.180,90.000,1591.259021,8.5900,11.4530,16.443713,90.000000,100.383702,90.000,1
22,AAMTXP,"(a,a)-bis(4,6-Dimethyl-2-thioxo-1,3,2-dioxapho...",False,True,False,1,True,False,NaN,NaN,...,113.380,90.000,1673.976496,9.8870,12.3250,14.966000,113.380000,90.000000,90.000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1436103,ZUVJOS01,"tris(2,4,6-tribromophenyl)methyl radical",False,True,False,1,True,False,NaN,red,...,100.904,90.000,2359.166565,8.4619,14.4325,19.672600,90.000000,100.904000,90.000,3
1436106,ZUYBOM01,"2-(4-chlorophenyl)-1,3-benzothiazole",False,True,False,1,True,False,NaN,colorless,...,98.556,90.000,1101.366516,7.1466,11.0497,14.104000,90.000000,90.000000,98.556,2
1436112,ZZZEIG05,triphenylarsane,False,True,False,4,True,False,NaN,whiteish colorless,...,80.072,86.361,2999.830947,11.2004,15.2918,17.882400,84.551000,80.072000,86.361,5
1436114,ZZZOLM02,docosanoic acid,False,True,False,1,True,False,NaN,colorless,...,90.052,90.000,2089.560807,5.5210,7.1540,52.904000,90.000000,90.052000,90.000,2


In [7]:
n_total = df_counted["inchi"].nunique()
n_mono = df_counted.groupby("inchi")["num_forms"].first().eq(1).sum()
n_poly = df_counted.groupby("inchi")["num_forms"].first().gt(1).sum()

print(f"Total molecules: {n_total}")
print(f"Monomorphs: {n_mono} ({round(100 * n_mono / n_total, 1)} %)")
print(f"Polymorphs: {n_poly} ({round(100 * n_poly / n_total, 1)} %)")

Total molecules: 382878
Monomorphs: 359864 (94.0 %)
Polymorphs: 23014 (6.0 %)


In [8]:
df_per_mol = df_counted.groupby("inchi").first()
false_monomorphs = df_per_mol[df_per_mol["polymorph"].isna() & (df_per_mol["num_forms"] > 1)]
n_mono_db = df_per_mol["polymorph"].isna().sum()

print(f"Molecules marked as monomorphs in DB but polymorphic by InChI: {len(false_monomorphs)}")
print(f"i.e. {round(100 * len(false_monomorphs) / n_mono_db, 1)}% of DB monomorphs")

Molecules marked as monomorphs in DB but polymorphic by InChI: 16307
i.e. 4.3% of DB monomorphs


In [9]:
df_counted.to_csv("csd_counted.csv", index=False)

### 4. Properties

In [9]:
entry = csd[0]
properties = dir(entry.crystal)
properties

['Contact',
 'Disorder',
 'HBond',
 'MillerIndices',
 'PeriodicBondChain',
 'ReducedCell',
 'Void',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_chemical_info',
 '_crystal',
 '_crystal_info',
 '_csv',
 '_decode_inclusion',
 '_identifier',
 '_voids_volume_telemetry',
 'add_hydrogens',
 'are_atoms_symmetry_related',
 'assign_bonds',
 'asymmetric_unit_molecule',
 'atoms_on_special_positions',
 'calculate_voids',
 'calculated_density',
 'cell_angles',
 'cell_lengths',
 'cell_volume',
 'centre_molecule',
 'contact_network',
 'contacts',
 'copy',
 'crystal_system',
 'disorder',
 'disordered_molecule',
 'formula',
 'fractional_to_orthogonal',
 'from_stri

In [ ]:
BANNED_LIST = ['melting_point_display_string', 'input_melting_point_range', 'formatted_melting_point_text', 'formatted_melting_point_range' ]
MAX_ENTRIES_TEST = 100

prop_list = {}
for prop in dir(entry):
    if prop.startswith('_') or prop in BANNED_LIST:
        continue

    try:
        attr_class = getattr(type(entry), prop, None)
        if isinstance(attr_class, property):
            doc = getattr(attr_class.fget, '__doc__', None)
        else:
            doc = getattr(getattr(entry, prop), '__doc__', None)

        doc_clean = doc.split('>>>')[0].split(':')[0].strip() if doc else None

        example_value = None
        for i, test_entry in enumerate(islice(csd, MAX_ENTRIES_TEST)):
            try:
                value = getattr(test_entry, prop)
                if value:
                    example_value = value
                    break
            except:
                continue

        # if str(example_value)[0] == '<':
        #     continue

        prop_list[prop] = {
            "description": doc_clean,
            "example": str(example_value) if example_value is not None else "No example found"
        }

    except Exception as e:
        if "Solubility Platform" not in str(e):
            print(f"Error with {prop} : {e}")

for prop, infos in prop_list.items():
    if infos['example'][0] == "<":
        print(f"--- {prop} ---")
        print(f"Description : {infos['description']}")
        print(f"Exemple : {infos['example']}")
        print()

print(len(prop_list))

In [ ]:
BANNED_LIST = ['melting_point_display_string', 'input_melting_point_range',
               'formatted_melting_point_text', 'formatted_melting_point_range']
MAX_ENTRIES_TEST = 100

entry = csd[0]
molecule = entry.molecule

prop_list = {}
for prop in dir(molecule):
    if prop.startswith('_') or prop in BANNED_LIST:
        continue

    try:
        attr_class = getattr(type(molecule), prop, None)
        if isinstance(attr_class, property):
            doc = getattr(attr_class.fget, '__doc__', None)
        else:
            doc = getattr(getattr(molecule, prop), '__doc__', None)

        doc_clean = doc.split('>>>')[0].split(':')[0].strip() if doc else None

        example_value = None
        for test_entry in islice(csd, MAX_ENTRIES_TEST):
            try:
                value = getattr(test_entry.molecule, prop)
                if value:
                    example_value = value
                    break
            except:
                continue

        # if str(example_value)[0] == '<':
        #     continue
        
        prop_list[prop] = {
            "description": doc_clean,
            "example": str(example_value) if example_value is not None else "No example found"
        }

    except Exception as e:
        if "Solubility Platform" not in str(e):
            print(f"Error with {prop} : {e}")

for prop, infos in prop_list.items():
    print(f"--- {prop} ---")
    print(f"Description : {infos['description']}")
    print(f"Exemple : {infos['example']}")
    print()

print(len(prop_list))